# PIC_SUB_02_check: data files and diagnosis-code matching

**Run this notebook on your own computer only.** PIC is credentialed data; nothing here is uploaded anywhere.

**Why.** PIC_SUB_02 found fewer admissions with an inclusion code than the feasibility counts of PIC_SUB_01 imply, and the two
runs read different files (PIC_SUB_01: uncompressed .csv; PIC_SUB_02: .csv.gz). This check describes the data files and how the
diagnosis codes match. It is outcome-blind: it does not apply the model and looks at no outcome, laboratory value or vital sign.

**How to run.** Put this notebook in the PIC_SUB_02 folder (it reads `pic_dir` from `PIC_SUB_02_config.txt`) and run all cells
(about 1 to 2 minutes). Send back the folder `output_pic_sub02_check/`. Counts of admissions and children are rounded to the
nearest 5 (1 to 4 shown as "<5"); file sizes, line counts and checksums are given exactly.

In [ ]:
import gzip, hashlib, json, math, platform, re, time
from pathlib import Path
import pandas as pd

T0 = time.time()
HERE = Path.cwd()
CONFIG = {}
_cfg = HERE / 'PIC_SUB_02_config.txt'
if _cfg.exists():
    for _line in _cfg.read_text(encoding='utf-8-sig').splitlines():
        if '=' in _line and not _line.strip().startswith('#'):
            _k, _v = _line.split('=', 1)
            CONFIG[_k.strip().lower()] = _v.strip().strip('"\'')
SRC = Path(CONFIG['pic_dir']) if CONFIG.get('pic_dir') else HERE
OUT = HERE / 'output_pic_sub02_check'
OUT.mkdir(exist_ok=True)
_ours = re.compile(r'^(0\d_[\w-]+\.csv|summary\.md|run_info\.json)$')
for _p in OUT.iterdir():
    if _p.is_file() and _ours.match(_p.name):
        _p.unlink()
AGE_MIN_DAYS, AGE_MAX_YEARS = 28, 18
ALIASES = {'D_DIAGNOSES_ICD': 'D_ICD_DIAGNOSES'}


def r5(n):
    """Counts of admissions or children: 0, '<5' for 1 to 4, otherwise rounded to the nearest 5 (as in PIC_SUB_02)."""
    n = int(n)
    return 0 if n == 0 else ('<5' if n < 5 else int(5 * math.floor(n / 5 + 0.5)))


def save(rows, name):
    df = pd.DataFrame(rows)
    df.to_csv(OUT / name, index=False, encoding='utf-8-sig')
    return df


def read(table, usecols=None):
    p = FILES[table]
    head = pd.read_csv(p, nrows=0)
    cmap = {c: c.strip().upper() for c in head.columns}
    cols = [c for c in head.columns if usecols is None or cmap[c] in usecols]
    return pd.read_csv(p, usecols=cols, dtype=str, keep_default_na=False, na_values=[''], encoding='utf-8').rename(columns=cmap)


def pick(cols, *names, contains=None):
    cols = list(cols)
    for n in names:
        if n in cols:
            return n
    if contains:
        for c in cols:
            if all(k in c for k in contains):
                return c
    return None


print('PIC folder:', SRC.name)

## Files
Size, SHA-256 and line count (lines minus the header line, as in PIC_SUB_01) of every table, and the comparison with
`SHA256SUMS.txt` when the folder has one (PhysioNet downloads include it).

In [ ]:
FILES, BOTH = {}, []
for p in sorted(SRC.iterdir()):
    m = re.match(r'^(.*?)\.csv(\.gz)?$', p.name, flags=re.I)
    if m:
        t = ALIASES.get(m.group(1).upper(), m.group(1).upper())
        if t in FILES:
            BOTH.append(t)
        FILES[t] = p                       # as in PIC_SUB_02: with both .csv and .csv.gz present, the later name (.csv.gz) is read
# SHA256SUMS.txt in this folder or up to two folders above it (PhysioNet keeps it at the top of the download).
SUMS, SUMS_DIR = {}, None
for _d in (SRC, SRC.parent, SRC.parent.parent):
    _f = next((_d / n for n in ('SHA256SUMS.txt', 'sha256sums.txt', 'SHA256SUMS') if (_d / n).is_file()), None)
    if _f:
        SUMS_DIR = _d
        for _line in _f.read_text(encoding='utf-8', errors='replace').splitlines():
            _parts = _line.strip().split(maxsplit=1)
            if len(_parts) == 2 and re.fullmatch(r'[0-9a-fA-F]{64}', _parts[0]):
                SUMS[_parts[1].strip().lstrip('*').replace('\\', '/').lstrip('./')] = _parts[0].lower()
        break
try:
    REL = SRC.resolve().relative_to(SUMS_DIR.resolve()).as_posix() if SUMS_DIR else ''
except ValueError:
    REL = ''
REL = '' if REL == '.' else REL
SUMS_HERE = {k.rsplit('/', 1)[-1]: v for k, v in SUMS.items() if (k.rsplit('/', 1)[0] if '/' in k else '') == REL}
SUMS_WHERE = 'absent' if not SUMS_DIR else ('in this folder' if not REL else f'{len(REL.split("/"))} folder(s) above')


def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 24), b''):
            h.update(b)
    return h.hexdigest()


def count_lines(p):
    opener = gzip.open if p.name.lower().endswith('.gz') else open
    with opener(p, 'rb') as f:
        return max(sum(b.count(b'\n') for b in iter(lambda: f.read(1 << 24), b'')) - 1, 0)


file_rows = []
for t, p in sorted(FILES.items()):
    h = sha256(p)
    status = 'no SHA256SUMS.txt' if not SUMS else ('match' if SUMS_HERE.get(p.name) == h else ('MISMATCH' if p.name in SUMS_HERE else 'not listed'))
    file_rows.append({'table': t, 'file': p.name, 'bytes': p.stat().st_size, 'lines_minus_header': count_lines(p),
                      'columns': ', '.join(c.strip().upper() for c in pd.read_csv(p, nrows=0).columns), 'sha256': h,
                      'sha256sums_txt': status})
for _name in sorted(set(SUMS_HERE) - {r['file'] for r in file_rows}):
    if re.search(r'\.csv(\.gz)?$', _name, flags=re.I):
        file_rows.append({'table': '', 'file': _name, 'bytes': '', 'lines_minus_header': '', 'columns': '', 'sha256': SUMS_HERE[_name],
                          'sha256sums_txt': 'listed for this folder but missing'})
FILES_DF = save(file_rows, '01_files.csv')
SUMS_FOLDERS = sorted({k.rsplit('/', 1)[0] if '/' in k else '(top level)' for k in SUMS if re.search(r'\.csv(\.gz)?$', k, flags=re.I)})
print(f'SHA256SUMS.txt: {SUMS_WHERE}; folders with table files listed in it: {SUMS_FOLDERS}; this folder: {REL or "(top level)"}')
if BOTH:
    print('Both .csv and .csv.gz present for:', BOTH, '(PIC_SUB_02 reads the .csv.gz)')
print(FILES_DF[['table', 'file', 'bytes', 'lines_minus_header', 'sha256sums_txt']].to_string(index=False))

## Diagnosis codes
The inclusion and exclusion lists are copied unchanged from PIC_SUB_02. Each code is matched two ways: exactly (as in
PIC_SUB_02, after trimming spaces at both ends) and normalized (all spaces removed, upper case). Raw spellings that differ from
the listed code are shown.

In [ ]:
LIST_A = {  # acute encephalopathy: code -> (Chinese title in the PIC dictionary, group)
    'G93.400': ('脑病', 'Encephalopathy codes'), 'G92.x00': ('中毒性脑病', 'Encephalopathy codes'),
    'G93.700': ('赖氏综合征', 'Encephalopathy codes'), 'G93.701': ('赖氏综合症', 'Encephalopathy codes')}
LIST_B = {  # acute encephalitis
    'G04.913': ('脑膜脑炎', 'Meningoencephalitis or encephalitis, unspecified'),
    'G04.914': ('脑膜脑炎', 'Meningoencephalitis or encephalitis, unspecified'),
    'G04.916': ('脑炎', 'Meningoencephalitis or encephalitis, unspecified'),
    'A86.x00': ('病毒性脑炎', 'Viral encephalitis'), 'A86.02': ('病毒性脑炎', 'Viral encephalitis'),
    'A86.01': ('病毒性脑膜脑炎', 'Viral encephalitis'), 'A86.x02': ('病毒性脑膜脑炎', 'Viral encephalitis'),
    'A86.x01': ('病毒性脑脊髓炎', 'Viral encephalitis'), 'A83.001': ('流行性乙型脑炎(日本脑炎)', 'Viral encephalitis'),
    'G04.921': ('脑干脑炎', 'Brainstem or focal encephalitis, encephalomyelitis'),
    'G04.910': ('脑干脑炎', 'Brainstem or focal encephalitis, encephalomyelitis'),
    'G04.908': ('局限性脑炎', 'Brainstem or focal encephalitis, encephalomyelitis'),
    'G04.909': ('局灶性脑炎', 'Brainstem or focal encephalitis, encephalomyelitis'),
    'G04.912': ('脑脊髓炎', 'Brainstem or focal encephalitis, encephalomyelitis'),
    'G04.000': ('播散性脑炎', 'ADEM or disseminated encephalitis'), 'G04.001': ('急性播散性脑脊髓炎', 'ADEM or disseminated encephalitis'),
    'G04.201': ('化脓性脑膜脑炎', 'Bacterial meningoencephalitis'), 'G04.807': ('化脓性脑膜脑炎', 'Bacterial meningoencephalitis')}
LIST_C = {  # exclusion codes: code -> (Chinese title, reason)
    'P91.600': ('新生儿缺氧缺血性脑病', 'neonatal condition'), 'P91.900': ('新生儿脑病', 'neonatal condition'),
    'P21.902': ('新生儿缺血缺氧性脑病', 'neonatal condition'), 'P59.901': ('新生儿高胆红素血症性脑病', 'neonatal condition'),
    'G93.101': ('缺氧性脑病（肺性脑病）', 'hypoxic or anoxic injury'), 'K72.903': ('肝昏迷(肝性脑病)', 'hepatic encephalopathy'),
    'I67.400': ('高血压性脑病', 'hypertensive encephalopathy'), 'I67.401': ('高血压性脑病', 'hypertensive encephalopathy'),
    'E16.107+': ('低血糖昏迷性脑病', 'metabolic encephalopathy'), 'E16.108+': ('低血糖性脑病', 'metabolic encephalopathy'),
    'G93.403': ('代谢性脑病', 'metabolic encephalopathy'), 'E51.200+': ('韦尼克脑病', 'metabolic encephalopathy'),
    'E51.201': ('韦尼克氏脑病(急性出血性脑灰质炎)', 'metabolic encephalopathy'), 'N18.801': ('尿毒症性脑病', 'metabolic encephalopathy'),
    'N18.806+': ('尿毒症性脑病', 'metabolic encephalopathy'), 'A81.200': ('进行性多灶性白质脑病', 'leukoencephalopathy'),
    'A81.201': ('播散性坏死性脑白质病', 'leukoencephalopathy'), 'I67.300': ('进行性血管性脑白质病', 'leukoencephalopathy'),
    'I67.301': ('宾斯旺格病', 'leukoencephalopathy'), 'G93.406': ('脑白质病', 'leukoencephalopathy'),
    'B94.100': ('病毒性脑炎后遗症', 'sequelae'), 'B94.101': ('病毒性脑炎后遗症', 'sequelae'), 'B94.102': ('流行性乙型脑炎后遗症', 'sequelae'),
    'F48.909': ('脑病后遗症', 'sequelae'), 'G09.02': ('脑炎后遗症', 'sequelae'), 'G09.x01': ('感染中毒性脑病后遗症', 'sequelae'),
    'T97.02': ('感染中毒后脑病后遗症', 'sequelae'), 'T97.x02': ('中毒性脑病后遗症', 'sequelae'),
    'G93.402': ('低颅压综合症', 'not an acute encephalopathy'), 'F07.201': ('创伤性脑病', 'not an acute encephalopathy'),
    'G93.407': ('脑病变', 'not an acute encephalopathy'), 'M32.114+': ('狼疮性脑病', 'not an acute encephalopathy'),
    'G06.003': ('手术后颅内感染', 'post-operative intracranial infection'), 'G04.801': ('化脓性脊髓炎', 'myelitis without encephalitis'),
    'G04.903': ('急性脊髓炎', 'myelitis without encephalitis'), 'G04.911': ('脑脊髓神经根神经炎', 'myelitis without encephalitis'),
    'G04.805': ('化学性脑膜炎', 'chemical meningitis'), 'A17.801': ('结核性脑膜脑炎', 'tuberculosis of the nervous system'),
    'A17.803+': ('结核性脑膜脑炎', 'tuberculosis of the nervous system'), 'A17.804': ('结核性脑炎', 'tuberculosis of the nervous system'),
    'A17.804+': ('结核性脑炎', 'tuberculosis of the nervous system')}
EXCL_PREFIX = r'^(A17|G06|P91)'     # tuberculosis of the nervous system, intracranial abscess and granuloma, neonatal cerebral disorders


def norm(s):
    return re.sub(r'\s+', '', str(s)).upper()


_missing = [t for t in ('PATIENTS', 'ADMISSIONS', 'DIAGNOSES_ICD', 'D_ICD_DIAGNOSES') if t not in FILES]
if _missing:
    raise FileNotFoundError(f'Tables not found in {SRC.name}: {_missing}. Point pic_dir to the folder that holds the PIC table files.')

PAT = read('PATIENTS', ['SUBJECT_ID', 'DOB'])
ADM = read('ADMISSIONS', ['SUBJECT_ID', 'HADM_ID', 'ADMITTIME'])
ADM = ADM.merge(PAT.assign(DOB=pd.to_datetime(PAT.DOB, errors='coerce')), on='SUBJECT_ID', how='left')
ADM['ADMITTIME'] = pd.to_datetime(ADM.ADMITTIME, errors='coerce')
ADM['AGE_DAYS'] = (ADM.ADMITTIME - ADM.DOB).dt.total_seconds() / 86400
ADM = ADM.drop_duplicates('HADM_ID').set_index('HADM_ID', drop=False)
ADM.index.name = None
AGED = set(ADM.index[ADM.AGE_DAYS.ge(AGE_MIN_DAYS) & ADM.AGE_DAYS.lt(AGE_MAX_YEARS * 365.25)])

DXD = read('D_ICD_DIAGNOSES')
C_CODE = pick(DXD.columns, 'ICD10_CODE_CN', contains=['CODE', 'CN']) or pick(DXD.columns, 'ICD10_CODE', contains=['CODE'])
C_TCN = pick(DXD.columns, 'TITLE_CN', contains=['TITLE', 'CN'])
C_TEN = pick([c for c in DXD.columns if c != C_TCN], 'TITLE', 'LONG_TITLE', contains=['TITLE'])
DXD = pd.DataFrame({'RAW': DXD[C_CODE].astype(str), 'TITLE_CN': DXD[C_TCN] if C_TCN else '', 'TITLE': DXD[C_TEN] if C_TEN else ''})
DXD['STRIP'], DXD['NORM'] = DXD.RAW.str.strip(), DXD.RAW.map(norm)
DICT_STRIP, DICT_NORM = set(DXD.STRIP), set(DXD.NORM)
TITLE_BY_NORM = DXD.drop_duplicates('NORM').set_index('NORM')[['TITLE_CN', 'TITLE']]

DX0 = read('DIAGNOSES_ICD')
DX_COLUMNS = list(DX0.columns)
C_DXCODE = pick(DX0.columns, 'ICD10_CODE_CN', contains=['CODE', 'CN']) or pick(DX0.columns, 'ICD10_CODE', contains=['CODE'])
C_CAT = pick(DX0.columns, 'DIAG_CATEGORY', contains=['CATEGORY'])
DX = DX0[['HADM_ID', C_DXCODE] + ([C_CAT] if C_CAT else [])].rename(columns={C_DXCODE: 'RAW', **({C_CAT: 'CAT'} if C_CAT else {})})
N_DX_ROWS, N_DX_NOCODE = len(DX), int(DX.RAW.isna().sum())
DX = DX.dropna(subset=['RAW']).copy()
DX['STRIP'], DX['NORM'] = DX.RAW.astype(str).str.strip(), DX.RAW.map(norm)
IN_ADM = DX.HADM_ID.isin(ADM.index)


def adm_counts(mask):
    h = set(DX.loc[mask & IN_ADM, 'HADM_ID'])
    return len(h), len(h & AGED)


code_rows = []
for lst, d in (('A', LIST_A), ('B', LIST_B), ('C', LIST_C)):
    for c, (title, grp) in d.items():
        ea, eg = adm_counts(DX.STRIP.eq(c))
        na, ng = adm_counts(DX.NORM.eq(norm(c)))
        raw = DX.loc[DX.NORM.eq(norm(c)), 'RAW'].value_counts()
        other = [f'{s!r} ({r5(k)})' for s, k in raw.items() if s.strip() != c]
        code_rows.append({'list': lst, 'code': c, 'group_or_reason': grp, 'in_dictionary_exact': c in DICT_STRIP,
                          'in_dictionary_normalized': norm(c) in DICT_NORM,
                          'admissions_exact_all_ages_n': r5(ea), 'admissions_exact_aged_28d_to_17y_n': r5(eg),
                          'admissions_normalized_all_ages_n': r5(na), 'admissions_normalized_aged_28d_to_17y_n': r5(ng),
                          'other_raw_spellings': '; '.join(other)})
CODES_DF = save(code_rows, '02_list_codes.csv')
print(CODES_DF[CODES_DF.list.ne('C')][['code', 'admissions_exact_aged_28d_to_17y_n', 'admissions_normalized_aged_28d_to_17y_n',
                                        'other_raw_spellings']].to_string(index=False))

## Related codes not on the lists
Codes used in DIAGNOSES_ICD that are not on any list but belong to the encephalitis and encephalopathy code families, or whose
dictionary title names encephalitis or encephalopathy (for review only).

In [ ]:
LISTED = {norm(c) for c in {**LIST_A, **LIST_B, **LIST_C}}
FAMILY = re.compile(r'^(G04|G05|G92|G93\.[147]|G94|A8[0-9]|B00\.4|B01\.1|B02\.0|B05\.0|B26\.2|A17|G06|P91)')
TITLE_TERMS = re.compile(r'脑炎|脑病|脑脊髓炎|encephal', flags=re.I)
CODE_LIKE = re.compile(r'^[A-Z]\d{2}(\.[0-9A-Z]{1,4})?[+*]?$')
near = []
for nc, g in DX[~DX.NORM.isin(LISTED)].groupby('NORM'):
    if not CODE_LIKE.match(nc):
        continue
    tcn, ten = (TITLE_BY_NORM.loc[nc, 'TITLE_CN'], TITLE_BY_NORM.loc[nc, 'TITLE']) if nc in TITLE_BY_NORM.index else ('', '')
    if FAMILY.match(nc) or TITLE_TERMS.search(f'{tcn} {ten}'):
        h = set(g.loc[g.HADM_ID.isin(ADM.index), 'HADM_ID'])
        near.append({'code_normalized': nc, 'raw_spellings': '; '.join(repr(s) for s in sorted(set(g.RAW))[:5]), 'title_cn': tcn,
                     'title_en': ten, 'in_dictionary': nc in DICT_NORM, 'admissions_all_ages_n': r5(len(h)),
                     'admissions_aged_28d_to_17y_n': r5(len(h & AGED)), '_k': len(h)})
NEAR_DF = pd.DataFrame(near, columns=['code_normalized', 'raw_spellings', 'title_cn', 'title_en', 'in_dictionary', 'admissions_all_ages_n',
                                      'admissions_aged_28d_to_17y_n', '_k'])
NEAR_DF = NEAR_DF.sort_values(['_k', 'code_normalized'], ascending=[False, True]).drop(columns='_k')
save(NEAR_DF, '03_related_codes.csv')
print(NEAR_DF.head(40).to_string(index=False))

## DIAGNOSES_ICD overall

In [ ]:
per_adm = DX[IN_ADM].groupby('HADM_ID').size()
bins = pd.cut(per_adm, [0, 1, 2, 3, 5, 10, 10 ** 6], labels=['1', '2', '3', '4-5', '6-10', '>10'])
overall = [
    ('columns', ', '.join(DX_COLUMNS)),
    ('rows', N_DX_ROWS),
    ('rows_without_code', N_DX_NOCODE),
    ('admissions_in_ADMISSIONS', len(ADM)),
    ('admissions_with_any_diagnosis_n', r5(per_adm.size)),
    ('admission_numbers_in_DIAGNOSES_ICD_not_in_ADMISSIONS_n', r5(DX.loc[~IN_ADM, 'HADM_ID'].nunique())),
    ('rows_code_not_in_dictionary_exact', int((~DX.STRIP.isin(DICT_STRIP)).sum())),
    ('rows_code_not_in_dictionary_normalized', int((~DX.NORM.isin(DICT_NORM)).sum())),
    ('rows_whose_trimmed_code_differs_from_raw', int((DX.RAW.astype(str) != DX.STRIP).sum())),
    ('rows_with_lowercase_letters_in_code', int(DX.STRIP.str.contains(r'[a-z]', regex=True).sum())),
    ('dictionary_codes', len(DXD)),
    ('dictionary_codes_with_lowercase_letters', int(DXD.STRIP.str.contains(r'[a-z]', regex=True).sum())),
]
overall += [(f'admissions_with_{k}_diagnosis_rows_n', r5(v)) for k, v in bins.value_counts().reindex(bins.cat.categories).fillna(0).items()]
if C_CAT:
    overall += [(f'rows_DIAG_CATEGORY_{k}', int(v)) for k, v in DX.CAT.fillna('(empty)').value_counts().head(10).items()]
OVERALL_DF = save([{'measure': m, 'value': v} for m, v in overall], '04_diagnoses_overall.csv')
print(OVERALL_DF.to_string(index=False))

## Cohort counts with exact and with normalized matching
The steps of PIC_SUB_02 up to the age criteria (the test criterion is not repeated here).

In [ ]:
def flow(match):
    inc = {norm(c) for c in {**LIST_A, **LIST_B}} if match == 'normalized' else set(LIST_A) | set(LIST_B)
    exc = {norm(c) for c in LIST_C} if match == 'normalized' else set(LIST_C)
    col = 'NORM' if match == 'normalized' else 'STRIP'
    inc_adm = set(DX.loc[DX[col].isin(inc), 'HADM_ID']) & set(ADM.index)
    exc_adm = set(DX.loc[DX[col].isin(exc) | DX[col].str.contains(EXCL_PREFIX, regex=True), 'HADM_ID'])
    A = ADM.loc[sorted(inc_adm)]
    steps = [('Admissions with an inclusion code', len(A))]
    A = A.sort_values(['SUBJECT_ID', 'ADMITTIME', 'HADM_ID']).drop_duplicates('SUBJECT_ID')
    steps.append(('First such admission per child (index admission)', len(A)))
    A = A[~A.index.isin(exc_adm)]
    steps.append(('No exclusion code in the index admission', len(A)))
    A = A[A.AGE_DAYS.ge(AGE_MIN_DAYS)]
    steps.append(('Aged 28 days or older', len(A)))
    A = A[A.AGE_DAYS.lt(AGE_MAX_YEARS * 365.25)]
    steps.append(('Younger than 18 years', len(A)))
    return steps


FL = {m: flow(m) for m in ('exact', 'normalized')}
flow_rows = [{'step': i + 1, 'description': d_, 'exact_n': r5(FL['exact'][i][1]), 'normalized_n': r5(FL['normalized'][i][1])}
             for i, (d_, _) in enumerate(FL['exact'])]
FLOW_DF = save(flow_rows, '05_flow_exact_vs_normalized.csv')
print(FLOW_DF.to_string(index=False))

In [ ]:
# Summary, run record and a privacy check of this folder (no identifier columns; admission counts rounded).
L_ = ['# PIC_SUB_02_check summary (outcome-blind; admission counts rounded to the nearest 5, 1-4 shown as <5)', '',
      f'Run {time.strftime("%d %b %Y %H:%M")}, {(time.time() - T0) / 60:.1f} min; python {platform.python_version()}, pandas {pd.__version__}.',
      f'PIC folder: {SRC.name} ({REL or "top level"} of the download); SHA256SUMS.txt: {SUMS_WHERE}; folders with table files '
      f'listed in it: {", ".join(SUMS_FOLDERS) or "none"}.',
      f'Tables present as both .csv and .csv.gz: {", ".join(BOTH) or "none"}.', '', '## Files', '',
      '| table | file | lines minus header | SHA256SUMS.txt |', '|---|---|---|---|']
L_ += [f"| {r['table']} | {r['file']} | {r['lines_minus_header']} | {r['sha256sums_txt']} |" for r in file_rows]
L_ += ['', '## Cohort steps, exact vs normalized code matching', '', '| step | description | exact | normalized |', '|---|---|---|---|']
L_ += [f"| {r['step']} | {r['description']} | {r['exact_n']} | {r['normalized_n']} |" for r in flow_rows]
L_ += ['', '## DIAGNOSES_ICD', '', '| measure | value |', '|---|---|'] + [f'| {m} | {v} |' for m, v in overall]
L_ += ['', 'Details: `02_list_codes.csv` (every listed code), `03_related_codes.csv` (codes not on the lists).',
       'Nothing here is computed by subphenotype or outcome; the model is not applied.']
(OUT / 'summary.md').write_text('\n'.join(L_) + '\n', encoding='utf-8')
(OUT / 'run_info.json').write_text(json.dumps({'notebook': 'PIC_SUB_02_check', 'pic_folder': SRC.name, 'python': platform.python_version(),
                                               'pandas': pd.__version__, 'minutes': round((time.time() - T0) / 60, 1)}, indent=1),
                                   encoding='utf-8')
ID_COL = re.compile(r'SUBJECT|HADM|ICUSTAY|ROW_ID|PATIENT', flags=re.I)
COUNT_OK = re.compile(r'^(|0|<5|\d*[05])$')
problems = []
for p in OUT.iterdir():
    if p.suffix == '.csv' and _ours.match(p.name):
        df = pd.read_csv(p, dtype=str, keep_default_na=False)
        problems += [f'{p.name}: identifier column {c}' for c in df.columns if ID_COL.search(c)]
        for c in [c for c in df.columns if c.endswith('_n')]:
            problems += [f'{p.name}: {c} value {v!r} not rounded' for v in df[c] if not COUNT_OK.match(v)]
        if 'measure' in df.columns:
            problems += [f'{p.name}: {m} value {v!r} not rounded' for m, v in zip(df.measure, df.value) if m.endswith('_n') and not COUNT_OK.match(v)]
print('Privacy check:', 'passed' if not problems else problems)
print('\n'.join(L_))